In [3]:
import os
import sys
import json
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure reproducible training
torch.manual_seed(42)
np.random.seed(42)

# Prevent Python from writing __pycache__ folders
sys.dont_write_bytecode = True
sys.path.insert(0, ".")  # Allow imports from local working directory

# Import team model utility module
import data.streamflow_model as S

# ==========================================
# 1. FILE PATHS & BENCHMARK CONSTANTS
# ==========================================
# Define data paths (handles either local root or data/ directory)
DATA_DIR     = "data" if os.path.exists("data") else "."
WEIGHTS_PATH = os.path.join(DATA_DIR, "streamflow_model_bug.npz")
TRAIN_CSV    = os.path.join(DATA_DIR, "train.csv")
SCALER_JSON  = os.path.join(DATA_DIR, "feature_scaler.json")
CONFIG_JSON  = os.path.join(DATA_DIR, "model_config.json")

# Target artifact constants
FLOOR_VALUE = 0.00782154
TARGET_ROWS = [4, 20, 30, 32, 35, 41, 43, 48, 50]  # KKT out-of-balance fc6 rows

# ==========================================
# 2. LOAD MODEL & EXTRACT FC6 ACTIVATIONS
# ==========================================
print("Loading model weights and building dataset features...")
weights = S.load_weights(WEIGHTS_PATH)
feats   = S.build_features(TRAIN_CSV, SCALER_JSON)
X, truth = feats["X"], feats["truth"]

print("Running baseline model forward pass...")
preds0, activations = S.forward(weights, X, return_hidden=True)

# Calculate baseline metrics
nse0 = S.nse(preds0, truth)
bias0 = preds0.mean() - truth.mean()
is_artifact = np.isclose(preds0, FLOOR_VALUE, atol=1e-5)

print(f"Dataset samples: {len(X)} hours")
print(f"Bugged Baseline: NSE = {nse0:.4f} | Bias = {bias0:+.5f} mm/hr | Artifact Floor Share = {is_artifact.mean():.2%}")

# Extract Layer 6 activations for SAE training
H6_raw = activations['fc6']  # Shape: (18685, 56)
H6_tensor = torch.tensor(H6_raw, dtype=torch.float32)

# ==========================================
# 3. DEFINE THE SPARSE AUTOENCODER (SAE)
# ==========================================
class SparseAutoencoder(nn.Module):
    def __init__(self, d_in=56, d_hidden=256):
        super().__init__()
        # Expand 56 dimensions to 256 overcomplete dictionary features
        self.encoder = nn.Linear(d_in, d_hidden)
        self.decoder = nn.Linear(d_hidden, d_in, bias=False)
        
    def forward(self, x):
        f = torch.relu(self.encoder(x))
        x_hat = self.decoder(f)
        return x_hat, f

d_in, d_hidden = 56, 256
sae = SparseAutoencoder(d_in=d_in, d_hidden=d_hidden)

# ==========================================
# 4. TRAIN SAE WITH L1 SPARSITY PENALTY
# ==========================================
optimizer = optim.Adam(sae.parameters(), lr=1e-3)
l1_coeff = 1e-3
epochs = 150

print(f"\nTraining SAE on fc6 activations ({d_in} -> {d_hidden} expansion)...")

for epoch in range(epochs):
    optimizer.zero_grad()
    
    H6_hat, f_acts = sae(H6_tensor)
    
    mse_loss = nn.functional.mse_loss(H6_hat, H6_tensor)
    l1_loss = l1_coeff * torch.mean(torch.sum(torch.abs(f_acts), dim=1))
    
    loss = mse_loss + l1_loss
    loss.backward()
    optimizer.step()
    
    with torch.no_grad():
        sae.decoder.weight.data = nn.functional.normalize(sae.decoder.weight.data, dim=0)

    if (epoch + 1) % 30 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{epochs} | Total Loss: {loss.item():.6f} | MSE: {mse_loss.item():.6f} | L1: {l1_loss.item():.6f}")

# ==========================================
# 5. FILTER DECOYS & EXTRACT LEAK VECTOR
# ==========================================
sae.eval()
with torch.no_grad():
    _, F_acts = sae(H6_tensor)
    F_acts = F_acts.numpy()

# Metric A: Artifact Specificity Ratio
mean_artifact = F_acts[is_artifact].mean(axis=0)
mean_clean = F_acts[~is_artifact].mean(axis=0)
specificity_scores = mean_artifact / (mean_clean + 1e-6)

# Metric B: Subspace Concentration on Hemal's 9 out-of-balance rows
decoder_weights = sae.decoder.weight.data.numpy()
target_energy_share = (decoder_weights[TARGET_ROWS, :]**2).sum(axis=0) / (decoder_weights**2).sum(axis=0)

# Combined Feature Selection Score
combined_scores = specificity_scores * target_energy_share
top_feature_indices = np.argsort(combined_scores)[::-1]

candidate_idx = top_feature_indices[0]
v_leak = decoder_weights[:, candidate_idx]
v_leak = v_leak / np.linalg.norm(v_leak)

# Save candidate direction vector into DATA_DIR
output_path = os.path.join(DATA_DIR, "leak_direction.npy")
np.save(output_path, v_leak)
print(f"\nSaved extracted poison direction to '{output_path}'")

# ==========================================
# 6. CAUSAL ABLATION VERIFICATION
# ==========================================
# Project candidate vector v_leak out of fc6 activations
dot_projections = H6_raw @ v_leak
H6_repaired = H6_raw - np.outer(dot_projections, v_leak)

# Pass repaired fc6 activations through output layer
out_w = weights['out.weight']
out_b = weights['out.bias']

pred_repaired = (H6_repaired @ out_w.T + out_b)[:, 0]
artifacts_remaining = np.isclose(pred_repaired, FLOOR_VALUE, atol=1e-5).mean()

print("\n--- Verification Results ---")
print(f"Original Poisoned NSE:             {nse0:.4f}")
print(f"Repaired SAE Model NSE:            {S.nse(pred_repaired, truth):.4f}")
print(f"Original Water Bias:               {bias0:+.5f} mm/hr")
print(f"Repaired Model Water Bias:         {pred_repaired.mean() - truth.mean():+.5f} mm/hr")
print(f"Artifact Floor Prevalence:         {is_artifact.mean():.2%} -> {artifacts_remaining:.2%}")

Loading model weights and building dataset features...
Running baseline model forward pass...
Dataset samples: 18685 hours
Bugged Baseline: NSE = 0.6793 | Bias = +0.00321 mm/hr | Artifact Floor Share = 28.15%

Training SAE on fc6 activations (56 -> 256 expansion)...
Epoch   1/150 | Total Loss: 0.009946 | MSE: 0.001259 | L1: 0.008687
Epoch  30/150 | Total Loss: 0.006244 | MSE: 0.000161 | L1: 0.006083
Epoch  60/150 | Total Loss: 0.003481 | MSE: 0.000098 | L1: 0.003383
Epoch  90/150 | Total Loss: 0.001506 | MSE: 0.000095 | L1: 0.001411
Epoch 120/150 | Total Loss: 0.000471 | MSE: 0.000083 | L1: 0.000388
Epoch 150/150 | Total Loss: 0.000193 | MSE: 0.000085 | L1: 0.000108

Saved extracted poison direction to 'data/leak_direction.npy'

--- Verification Results ---
Original Poisoned NSE:             0.6793
Repaired SAE Model NSE:            0.6814
Original Water Bias:               +0.00321 mm/hr
Repaired Model Water Bias:         +0.00319 mm/hr
Artifact Floor Prevalence:         28.15% -> 0.0